# bro-agent Training Worker (pull-based)

Runs **in this notebook** on a free GPU (Google Colab T4 **or** Kaggle GPU T4),
but is **driven by bro-agent**: it pulls fine-tune jobs from bro-agent's own
SQLite queue over HTTP, trains with Unsloth QLoRA, uploads the LoRA adapter
**back to bro-agent** (own infra — no Hugging Face, no Supabase, no Error Inbox),
marks the job done, then **wipes the local job dir** before the next job.

## Secrets needed (add before Run all)
- `BRO_BASE_URL` — e.g. https://bro-agent.onrender.com (no trailing slash)
- `BRO_ACCESS_CODE` — the bro-agent access code

Colab: Runtime secrets (key icon) or env vars. Kaggle: Add-ons → Secrets, and
turn **Internet ON**. Run all cells and leave it running — the loop polls every
30s. Note: bro-agent sleeps on Render's free tier; the first poll after idle
can take ~60s to wake it — the worker retries automatically. Restarting this
notebook loses nothing: jobs stay in bro-agent and get re-claimed.


In [ ]:
%%capture
import torch
major, _ = torch.cuda.get_device_capability()
pkg = "unsloth[colab-new]" if major >= 8 else "unsloth"
print("installing", pkg, "...")
# Canonical Unsloth install (Colab + Kaggle)
!pip install --no-deps "{pkg} @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes datasets requests
print("done — if `import unsloth` fails below, Runtime → Restart session, then re-run from the next cell.")


In [ ]:
import os, json, time, shutil, getpass
import requests

def get_secret(name):
    v = os.environ.get(name)
    if v:
        return v
    try:  # Google Colab runtime secrets
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    try:  # Kaggle secrets
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v:
            return v
    except Exception:
        pass
    return getpass.getpass(name + ": ")

BASE = get_secret("BRO_BASE_URL").rstrip("/")
CODE = get_secret("BRO_ACCESS_CODE")

def _post(path, payload=None, files=None, timeout=120):
    for attempt in range(4):  # Render free tier sleeps — retry the wake-up
        try:
            r = requests.post(BASE + path, json=payload, files=files, timeout=timeout)
            return r
        except requests.RequestException as e:
            print("request failed (attempt %d): %s" % (attempt + 1, e))
            time.sleep(15)
    return None

def claim_job():
    r = _post("/api/train/claim", {"access_code": CODE})
    if r is None:
        return None
    if r.status_code == 404:
        return None
    r.raise_for_status()
    return r.json()

def complete_job(job_id, status, result_url="", metrics=None, error=""):
    r = _post("/api/train/%s/complete" % job_id,
              {"access_code": CODE, "status": status, "result_url": result_url,
               "metrics": metrics or {}, "error": error or ""})
    return r is not None and r.status_code == 200

def upload_file(job_id, local_path):
    fname = os.path.basename(local_path)
    with open(local_path, "rb") as f:
        data = f.read()
    r = _post("/api/train/%s/upload" % job_id,
              payload=None,
              files={"file": (fname, data, "application/octet-stream"),
                     "access_code": (None, CODE)},
              timeout=600)
    ok = r is not None and r.status_code == 200
    print(("uploaded " if ok else "UPLOAD FAILED ") + fname)
    return ok

print("bro-agent:", BASE)
print("cuda:", __import__("torch").cuda.get_device_name(0))


In [ ]:
import os, torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

MAX_STEPS = int(os.environ.get("TS_MAX_STEPS", "60"))

def _format(ds, tokenizer):
    cols = set(ds.column_names)
    eos = tokenizer.eos_token or ""
    if "instruction" in cols:  # Alpaca-style
        def f(x):
            out = x.get("response", x.get("output", ""))
            if x.get("input"):
                t = "### Instruction:\n%s\n\n### Input:\n%s\n\n### Response:\n%s" % (
                    x.get("instruction", ""), x["input"], out)
            else:
                t = "### Instruction:\n%s\n\n### Response:\n%s" % (x.get("instruction", ""), out)
            return {"text": t + eos}
        return ds.map(f)
    if "messages" in cols:  # chat format
        def f(x):
            t = "".join("<%s>\n%s\n" % (m.get("role", "user"), m.get("content", ""))
                        for m in x["messages"])
            return {"text": t + eos}
        return ds.map(f)
    c0 = ds.column_names[0]  # fallback: first column as raw text
    return ds.map(lambda x: {"text": str(x[c0]) + eos})

def train_job(job, workdir):
    model_repo = job["model_repo"]
    dataset_repo = job.get("dataset")
    rank = int(job.get("lora_rank") or 16)
    os.makedirs(workdir, exist_ok=True)

    print("loading", model_repo)
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_repo, max_seq_length=2048,
        dtype=None, load_in_4bit=True)
    model = FastLanguageModel.get_peft_model(
        model, r=rank,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_alpha=rank * 2, lora_dropout=0, bias="none",
        use_gradient_checkpointing="unsloth", random_state=3407,
        use_rslora=True, loftq_config=None)

    if not dataset_repo:
        raise ValueError("job has no dataset — set one when queueing")
    print("loading dataset", dataset_repo)
    ds = load_dataset(dataset_repo, split="train")
    ds = _format(ds, tokenizer)
    print("examples:", len(ds))

    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, train_dataset=ds,
        dataset_text_field="text", max_seq_length=2048, dataset_num_proc=2,
        args=TrainingArguments(
            per_device_train_batch_size=2, gradient_accumulation_steps=4,
            warmup_steps=5, max_steps=MAX_STEPS, learning_rate=2e-4,
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=1, optim="adamw_8bit", weight_decay=0.01,
            lr_scheduler_type="cosine", seed=3407,
            output_dir=os.path.join(workdir, "outputs"),
        ),
    )
    trainer.train()
    train_loss = None
    for e in reversed(trainer.state.log_history):
        if "loss" in e:
            train_loss = float(e["loss"])
            break

    out_dir = os.path.join(workdir, "adapter")
    model.save_pretrained(out_dir)
    tokenizer.save_pretrained(out_dir)
    metrics = {"train_loss": train_loss, "examples": len(ds),
               "max_steps": MAX_STEPS, "lora_rank": rank}
    print("metrics:", metrics)
    return out_dir, metrics


In [ ]:
import time, shutil

POLL_SECS = int(os.environ.get("TS_POLL_SECS", "30"))

print("worker up — polling every %ds. Stop the cell to quit." % POLL_SECS)
while True:
    job = claim_job()
    if not job:
        time.sleep(POLL_SECS)
        continue
    jid = job["id"]
    workdir = "/tmp/ts-job-%s" % jid[:8]
    try:
        print("claimed", jid, job["model_repo"], "->", job.get("dataset"))
        out_dir, metrics = train_job(job, workdir)
        n_ok, n_fail = 0, 0
        for root, _, files in os.walk(out_dir):
            for fn in files:
                if upload_file(jid, os.path.join(root, fn)):
                    n_ok += 1
                else:
                    n_fail += 1
        result_url = "%s/api/train/%s/files/adapter_model.safetensors" % (BASE, jid)
        ok = complete_job(jid, "done" if n_fail == 0 else "failed",
                          result_url=result_url if n_fail == 0 else "",
                          metrics=metrics,
                          error=("%d files failed to upload" % n_fail) if n_fail else "")
        print("done", jid, "uploaded %d files, marked: %s" % (n_ok, ok))
    except Exception as e:
        import traceback; traceback.print_exc()
        complete_job(jid, "failed", error=str(e)[:2000])
        print("failed", jid, str(e)[:200])
    finally:
        # The user's rule: transport everything out, wipe local disk before next job.
        shutil.rmtree(workdir, ignore_errors=True)
        try:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:
            pass
        print("wiped", workdir)
